# Taller: Grafos en Ciencia de Datos

> *Dada una red de carreteras real, ¿desde qué punto conviene partir si quiero llegar lo más rápido posible a todos los demás lugares de la red?*

Esa pregunta aparentemente simple nos permitirá recorrer los conceptos fundamentales de la teoría de grafos, a construir un grafo a partir de tablas de datos reales, a calculardistancias mínimas sobre miles de nodos, y a visualizar el resultado de forma que cualquier persona pueda explorarlo interactivamente.

Los datos son una red de carreteras de Gran Bretaña: cientos de miles de nodos (intersecciones) conectados por aristas (tramos de carretera) con pesos que representan distancia o tiempo de viaje.

## 1. Conceptos fundamentales de grafos

Un grafo es una estructura matemática que modela relaciones entre objetos.

Formalmente, un grafo $G$ se define como un par $G = (V, E)$, donde:

- $V$ es un conjunto de **nodos** (o vértices): los objetos que queremos modelar.
- $E$ es un conjunto de **aristas** (o edges): las relaciones entre pares de nodos.

### Tipos de grafos

| Tipo | Descripción | Ejemplo |
|------|-------------|--------|
| **No dirigido** | Las aristas no tienen dirección; la relación es simétrica | Red de amistades |
| **Dirigido** (dígrafo) | Las aristas tienen dirección: $(u, v) \neq (v, u)$ | Twitter (seguir) |
| **Ponderado** | Cada arista tiene un peso numérico asociado | Carreteras (distancia, tiempo) |
| **No ponderado** | Todas las aristas valen igual | Hiperenlaces web |

En este taller trabajamos con un grafo no dirigido y ponderado: las carreteras pueden recorrerse en ambos sentidos, y cada tramo tiene un peso (longitud o tiempo de viaje).

### Conceptos clave

**Grado de un nodo (*degree*):** El número de aristas que inciden en un nodo. En un grafo no dirigido, indica cuántos vecinos directos tiene ese nodo. Un nodo con grado alto es una intersección muy conectada; con grado 1 es un callejón sin salida.

**Camino (*path*):** Una secuencia de nodos $v_1, v_2, \ldots, v_k$ tal que existe una arista entre cada par consecutivo. La longitud del camino es la suma de los pesos de sus aristas (en grafos ponderados) o simplemente el número de aristas (en no ponderados).

**Camino mínimo (*shortest path*):** El camino de menor coste total entre dos nodos. En una red de carreteras, puede significar la ruta más corta en kilómetros o la más rápida en tiempo.

**Distancia entre dos nodos:** La longitud del camino mínimo entre ellos. Si no hay camino (el grafo no es conexo), la distancia es infinita.

**Componente conexa:** Un subconjunto de nodos tal que existe un camino entre cualquier par. Una red de carreteras puede tener nodos aislados (islas sin conexión terrestre), que forman componentes separadas.

**Centralidad (*centrality*)** Familia de métricas que cuantifican la importancia de un nodo dentro de la red. Algunas de las más usadas son la centralidad de grado para nodos con más conexiones directas, centralidad de intermediación para nodos que aparecen en muchos caminos mínimos y cuya eliminación desconectaría la red, y PageRank para nodos que son importantes porque sus vecinos también lo son.


## 2. Herramientas: NetworkX, cuGraph y Graphistry

### NetworkX

[NetworkX](https://networkx.org/) es la librería de referencia para grafos en Python. Permite crear, manipular y analizar grafos con una API intuitiva, e incluye implementaciones de cientos de algoritmos. Es ideal para prototipado, enseñanza y redes de tamaño moderado (cientos de miles de nodos). Sin embargo, corre íntegramente en CPU con un solo hilo. En redes muy grandes (millones de nodos/aristas), ciertos algoritmos pueden tardar minutos o incluso horas.

```python
import networkx as nx
G = nx.Graph()                      # grafo no dirigido
G.add_edge('A', 'B', weight=10)
nx.shortest_path(G, 'A', 'B', weight='weight')
```

### cuGraph

[cuGraph](https://github.com/rapidsai/cugraph) es el equivalente GPU de NetworkX, parte del ecosistema RAPIDS. Así como cuDF acelera pandas moviéndola a GPU, cuGraph permite ejecutar los mismos algoritmos de grafos con aceleraciones de 10× a 100× en grafos grandes.

```python
import cugraph as cg
import cudf

edges = cudf.read_csv('road_graph.csv')
G = cg.Graph()
G.from_cudf_edgelist(edges, source='src', destination='dst', edge_attr='length')
resultado = cg.sssp(G, source=0)    # Single Source Shortest Path en GPU
```

### Graphistry

[Graphistry](https://www.graphistry.com/) es una plataforma de visualización interactiva de grafos e hipergrafos. A diferencia de las visualizaciones estáticas de NetworkX (basadas en matplotlib), Graphistry renderiza el grafo en el navegador con aceleración GPU, permitiendo explorar redes de millones de nodos con zoom, filtros y selección interactiva. El resultado es un widget embebible en el notebook con el que el usuario puede rotar, hacer zoom y seleccionar nodos de interés.

```python
import graphistry
graphistry.register(api=3, username='...', password='...')

graphistry\
    .edges(edges_df, 'src', 'dst')\
    .nodes(nodes_df, 'node_id')\
    .plot()
```


## 3. Dataset: red de carreteras de Gran Bretaña

Trabajamos con cuatro archivos CSV que juntos describen la red completa de carreteras de Gran Bretaña:

| Archivo | Columnas | Descripción |
|---------|----------|-------------|
| `road_nodes.csv` | `node_id`, `east`, `north`, `type` | Coordenadas geográficas de cada intersección |
| `road_graph.csv` | `src`, `dst`, `length` | Aristas con peso = longitud en metros |
| `road_graph_speed.csv` | `src`, `dst`, `length_s` | Aristas con peso = tiempo de viaje en segundos |
| `node_graph_map.csv` | `node_id`, `graph_id` | Mapeo entre IDs originales e IDs enteros del grafo |

Los archivos `road_graph` y `road_graph_speed` ya usan `graph_id` (enteros eficientes) en vez del `node_id` original (cadenas largas). El archivo `node_graph_map` permite traducir entre ambos cuando sea necesario.

La red contiene cientos de miles de nodos (intersecciones) y millones de aristas (tramos de carretera). Es una red real, con toda su complejidad: islas sin conexión terrestre, carreteras de un solo sentido representadas como pares de aristas, y loops donde una carretera vuelve al mismo nodo.

### Configuración de entorno

In [1]:
import sys

# Instalar cuda-python (componente esencial para interop CUDA)
!pip install cuda-python

# Determinar la versión de CUDA en Colab para instalar los paquetes correctos
cuda_major_version = !nvcc --version | grep -oP 'release \K\d+'
cuda_version_suffix = f"cu{cuda_major_version[0]}" if cuda_major_version else "cu12" # Default to cu12 if detection fails

print(f"Instalando RAPIDS para CUDA {cuda_version_suffix.replace('cu','')}.x...")
!pip install cudf-{cuda_version_suffix} cugraph-{cuda_version_suffix} --extra-index-url=https://pypi.nvidia.com

python_version_path = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages"
if python_version_path not in sys.path:
    sys.path.insert(0, python_version_path)

print("RAPIDS components (cuDF, cuGraph) installed.")

Instalando RAPIDS para CUDA 12.x...
Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 50.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 33.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 59.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 GB 28.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cugraph-cu12 to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 234.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 246.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 152.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 70.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━

In [22]:
import warnings
warnings.filterwarnings('ignore')

import cudf
import cugraph as cg
import pandas as pd
import networkx as nx
import numpy as np

### Conjunto de datos

In [3]:
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Talleres/GPU-Accelerated Graph Analytics/data"

import cudf

base_path = "/content/drive/MyDrive/Talleres/GPU-Accelerated Graph Analytics/data"
road_nodes = cudf.read_csv(f"{base_path}/road_nodes.csv", dtype=['str', 'float32', 'float32', 'str'])
road_graph = cudf.read_csv(f"{base_path}/road_graph.csv", dtype=['int32', 'int32', 'float32'])
speed_graph = cudf.read_csv(f"{base_path}/road_graph_speed.csv", dtype=['int32', 'int32', 'float32'])
node_map = cudf.read_csv(f"{base_path}/node_graph_map.csv")

Mounted at /content/drive


In [4]:
# Eliminamos duplicados en road_nodes (algunos nodos aparecen en varios tiles del mapa original)
road_nodes = road_nodes.drop_duplicates()

print('road_nodes:   ', road_nodes.shape)
print('road_graph:   ', road_graph.shape)
print('speed_graph:  ', speed_graph.shape)
print('node_map:     ', node_map.shape)

road_nodes:    (3078117, 4)
road_graph:    (7218169, 3)
speed_graph:   (7218169, 3)
node_map:      (3078117, 2)


In [5]:
road_nodes.head(3)

,node_id,east,north,type
0,id02FE73D4-E88D-4119-8DC2-6E80DE6F6594,320608.09375,870994.0000,junction
1,id634D65C1-C38B-4868-9080-2E1E47F0935C,320628.50000,871103.8125,road end
2,idDC14D4D1-774E-487D-8EDE-60B129E5482C,320635.46875,870983.8750,junction


In [6]:
road_graph.head(3)

,src,dst,length
0,0,129165,44.0
1,1,1678323,70.0
2,1,2372610,18.0


In [7]:
speed_graph.head(3)

,src,dst,length_s
0,0,129165,3.280848
1,1,1678323,5.219531
2,1,2372610,1.342165


## 4. Construcción del grafo

Una red de carreteras vive naturalmente en dos tablas:

- Una tabla de **nodos** (las intersecciones), con atributos como coordenadas.
- Una tabla de **aristas** (los tramos), con los IDs de origen y destino, más el peso.

Esa estructura (una lista de aristas con pesos) es exactamente lo que necesita cuGraph para construir el grafo. Es necesario indicar al modelo que cada fila de esta tabla es una conexión entre dos nodos.

La función `from_cudf_edgelist` lee exactamente eso:

In [8]:
import cupy as cp
import time

G = cg.Graph()

t0 = time.time()
G.from_cudf_edgelist(road_graph, source='src', destination='dst', edge_attr='length')
cp.cuda.Stream.null.synchronize()
t1 = time.time()

print(f'Construcción del grafo en GPU: {t1-t0:.3f} segundos')
print(f'Nodos en el grafo: {G.number_of_nodes():,}')
print(f'Aristas en el grafo: {G.number_of_edges():,}')

Construcción del grafo en GPU: 0.633 segundos
Nodos en el grafo: 3,078,117
Aristas en el grafo: 3,620,793


**Nota:** Estamos usando un `cg.Graph()` no dirigido, con el que cada fila de la tabla de aristas genera automáticamente dos conexiones (ida y vuelta). Por eso, si la tabla tiene $N$ filas, el grafo tendrá $2N$ aristas. Esto es apropiado para carreteras de doble sentido; si quisiéramos modelar calles de un solo sentido usaríamos `cg.DiGraph()`.

### Comparación con NetworkX en CPU

Para apreciar el valor de la aceleración GPU, construyamos el mismo grafo con NetworkX. En el entorno GPU-T4 de Google Colab, los tiempos seguirán siendo más que competentes, pero sirven como referencia para entender la amplia mejora que supone el uso de procesos en GPU.

> Estos bloques reconstruyen el grafo en CPU como un baseline para medir los tiempos de ejecución con diferente hardware; el resultado se desecha. El resto del taller usa cuGraph para un trabajo más eficiente.

In [9]:
# Construimos el grafo completo con NetworkX en CPU
road_graph_pd = road_graph.to_pandas()

print(f'Construyendo grafo NetworkX con {len(road_graph_pd):,} aristas...')
%time G_nx = nx.from_pandas_edgelist(road_graph_pd, source='src', target='dst', edge_attr='length')
print(f'Nodos: {G_nx.number_of_nodes():,} | Aristas: {G_nx.number_of_edges():,}')

Construyendo grafo NetworkX con 7,218,169 aristas...
CPU times: user 34 s, sys: 1.26 s, total: 35.3 s
Wall time: 35.6 s
Nodos: 3,078,117 | Aristas: 3,620,793


## 5. Análisis estructural de la red

Antes de responder la pregunta central, exploremos la estructura del grafo. Esto es análogo a hacer un `.describe()` en pandas: queremos entender qué tan conectada está la red, si hay nodos críticos, y si existen irregularidades.

### Distribución de grados

Primero, revisamos las estadísticas de grado de los nodos.

In [10]:
deg_df = G.degree()
print(deg_df['degree'].describe()[1:])

mean     4.689990
std      1.913452
min      2.000000
25%      2.000000
50%      6.000000
75%      6.000000
max     16.000000
Name: degree, dtype: float64


Recuerda que, en un grafo no dirigido, cada arista cuenta dos veces para el grado (una por cada extremo). Por eso todos los grados deberían ser múltiplos de 2. Para asegurarnos, verificamos que no haya nodos con grado impar.

In [11]:
nodos_grado_impar = deg_df[deg_df['degree'].mod(2) == 1]
print(f'Nodos con grado impar: {len(nodos_grado_impar)}')

Nodos con grado impar: 0


### Nodos más conectados

Los nodos con mayor grado son las intersecciones más importantes de la red: concentradores viales donde convergen muchas carreteras. Podemos revisar los más importantes.

In [12]:
deg_df.nlargest(10, 'degree')

,degree,vertex
0,16,652907
1,16,1252792
2,14,1606375
3,14,1826781
4,14,1990110
5,14,2122903
6,12,14774
7,12,48566
8,12,61891
9,12,332706


Al mismo tiempo, podemos formular consultas para saber, por ejemplo, cuántos nodos tienen un grado mínimo. Es decir, y considerando lo establecido durante los conceptos fundamentales, cuántos nodos son un dead-end o callejones sin salida.

In [13]:
min_degree = int(deg_df['degree'].min())
dead_ends = len(deg_df[deg_df['degree'] == min_degree])
print(f'Nodos con grado {min_degree} (extremos de carretera / callejones sin salida): {dead_ends:,}')
print(f'Representan el {100*dead_ends/len(deg_df):.1f}% de los nodos')

Nodos con grado 2 (extremos de carretera / callejones sin salida): 941,442
Representan el 30.6% de los nodos


### Loops en la red

Algunos tramos forman un loop: la carretera sale y vuelve al mismo nodo (una rotonda, por ejemplo).

In [14]:
loops = road_graph.loc[road_graph['src'] == road_graph['dst']]
print(f'Aristas en loop (src == dst): {len(loops)}')
if len(loops) > 0:
    print(loops.head())

Aristas en loop (src == dst): 23417
     src  dst  length
4      2    2    55.0
145   62   62   108.0
293  124  124    67.0
471  196  196    26.0
571  240  240    44.0


### Centralidad de intermediación sobre la muestra

En este caso, podemos calcular centralidad de intermediación. Trabajaremos sobre el grafo de NetworkX usando cuGraph como backend. Esto nos dice qué nodos actúan como puentes críticos en la red: si los eliminásemos, muchas rutas dejarían de existir.

En este caso, intentar calcular la centralidad de intermediación sobre el grafo completo (`G_nx`), que representa toda la red de carreteras de Gran Bretaña con millones de aristas, excede drásticamente la capacidad de memoria de la GPU, incluso utilizando cuGraph como backend. A pesar de la aceleración que ofrece cuGraph, hay límites físicos de memoria que no pueden ser superados si el grafo es demasiado grande.

Entonces, si el grafo completo es demasiado grande para la GPU, reducimos el grafo a 10,000 aristas para una nueva demostración.

In [16]:
# bc = nx.betweenness_centrality(G_nx, backend="cugraph")

# top_bc = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:5]
# for nodo, valor in top_bc:
#     print(f'  Nodo {nodo:>6}  →  {valor:.6f}')

In [17]:
sample_bc_edges = road_graph.sample(n=10000, random_state=42).to_pandas()
sample_G_nx_bc = nx.from_pandas_edgelist(sample_bc_edges, source='src', target='dst', edge_attr='length')

bc = nx.betweenness_centrality(sample_G_nx_bc, backend="cugraph")

top_bc = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:5]
for nodo, valor in top_bc:
    print(f'  Nodo {nodo:>6}  →  {valor:.6f}')

  Nodo 1981404  →  0.000000
  Nodo 696225  →  0.000000
  Nodo 1454746  →  0.000000
  Nodo 126187  →  0.000000
  Nodo 1695350  →  0.000000


Si bien un enfoque apropiado para estas operaciones sería probar calcular las métricas computacionalmente complejas construyendo un nuevo grafo más pequeño a partir de un subconjunto de datos con muestras reducidas, en este caso no es suficiente.

Incluso probando con alrededor de 10,000 aristas, los resultados arrojaron valores de centralidad de 0.0 para todos los nodos principales. Esto se debe a que, al reducir tan drásticamente el tamaño del grafo, se pierden demasiadas relaciones estructurales cruciales. Los caminos mínimos y las interconexiones que definen la intermediación de un nodo simplemente no se capturan con una muestra tan pequeña, haciendo que el algoritmo no encuentre nodos que actúen significativamente como puentes.

Para algoritmos tan demandantes, y con grafos de la magnitud del actual, es un desafío encontrar un equilibrio entre la manejabilidad computacional y la representatividad de la muestra.

## 6. Camino mínimo desde un solo origen (SSSP)

El algoritmo **Single Source Shortest Path (SSSP)** responde a la pregunta: dado un nodo de origen, ¿cuál es la distancia mínima hasta *todos* los demás nodos del grafo?

En el contexto de nuestra red de carreteras, lo aplicaremos dos veces:
1. Con el grafo ponderado por **longitud** (metros) para encontrar las rutas más cortas en distancia.
2. Con el grafo ponderado por **tiempo de viaje** (segundos) para encontrar las rutas más rápidas.

Haremos uso nuevamente de cuGraph, que implementa el algoritmo de Bellman-Ford generalizado sobre GPU, con lo que es capaz de calcular millones de caminos mínimos en segundos.

### Construcción del grafo de tiempos

Primero que todo, construimos el grafo ponderado por tiempo de viaje, en segundos.

In [18]:
G_speed = cg.Graph()
G_speed.from_cudf_edgelist(speed_graph, source='src', destination='dst', edge_attr='length_s')
print(f'Nodos: {G_speed.number_of_nodes():,} | Aristas: {G_speed.number_of_edges():,}')

Nodos: 3,078,117 | Aristas: 3,620,793


### Selección del nodo de partida

Elegimos el nodo con mayor grado como punto de partida: es la intersección más conectada de la red, posiblemente la que permita alcanzar el resto de la red más eficientemente.

In [19]:
deg_speed = G_speed.degree()
nodo_origen = deg_speed.nlargest(1, 'degree')['vertex'].iloc[0]
print(f'Nodo de origen seleccionado: {nodo_origen}')
print(f'Grado: {deg_speed[deg_speed["vertex"] == nodo_origen]["degree"].iloc[0]}')

Nodo de origen seleccionado: 652907
Grado: 16


### SSSP con pesos de distancia

A continuación, calculamos el SSSP desde un único nodo de origen hasta todos los demás nodos del grafo. En este caso, el grafo está ponderado por la longitud en metros de los segmentos de carretera, lo que significa que el algoritmo encontrará las rutas más cortas en términos de distancia física.

El resultado almacenado en `dist_metros` contendrá la distancia mínima para cada nodo alcanzable, lo que nos permite entender qué tan 'cerca' (en metros) está cada punto de la red del nodo de origen seleccionado.

In [20]:
dist_metros = cg.sssp(G, nodo_origen)
dist_metros.head()

,distance,vertex,predecessor
0,0.0,652907,-1
1,110322.0,1252792,1378560
2,213691.0,1606375,1375709
3,106434.0,1826781,2377706
4,253196.0,1990110,1652530


### Tratamiento de nodos inalcanzables

En algoritmos como SSSP, es común que no todos los nodos del grafo sean alcanzables desde el nodo de origen. Esto puede ocurrir si el grafo no es completamente conexo.

Para estos nodos inalcanzables, las implementaciones de grafos aceleradas por GPU, como `cugraph`, a menudo asignan un valor numérico muy grande en lugar de `Inf` o `NaN`. Específicamente, suelen usar el valor máximo representable por el tipo de dato flotante utilizado. Esto se hace por eficiencia computacional en la GPU, ya que `Inf` o `NaN` pueden requerir lógica de manejo especial que ralentiza las operaciones paralelas.

Es crucial filtrar estos valores antes de calcular estadísticas (como la media o el máximo de distancias), ya que de lo contrario sesgarían drásticamente los resultados, haciendo que parezca que existen distancias extremadamente grandes que en realidad solo indican falta de conexión. Al filtrarlos, nos aseguramos de que nuestras estadísticas reflejen solo las distancias reales dentro de la componente conexa del nodo de origen.

In [23]:
max_float32_val = np.finfo(np.float32).max
print(f'Nodos no alcanzables: {(dist_metros["distance"] >= max_float32_val).sum():,}')

Nodos no alcanzables: 29,446


In [24]:
dist_metros_validas = dist_metros['distance'].loc[dist_metros['distance'] < max_float32_val]
print('\nDistribución de distancias desde el nodo origen, excluyendo nodos desconectados:')
print(dist_metros_validas.describe()[1:])


Distribución de distancias desde el nodo origen, excluyendo nodos desconectados:
mean    210086.093750
std     137145.765625
min          0.000000
25%     125054.500000
50%     181815.500000
75%     252472.250000
max     868870.500000
Name: distance, dtype: float64


Una vez filtrados los nodos inalcanzables, la distribución de distancias desde el nodo de origen muestra un rango coherente de valores. La distancia media (210 km) y la distancia máxima (868 km) nos dan una idea de la extensión de la red desde el punto de partida seleccionado. Los cuartiles nos permiten entender la dispersión: el 75% de los nodos alcanzables están a menos de 252 km del origen.

### SSSP con pesos de tiempo

Ahora, repetimos el proceso de SSSP, pero utilizando el grafo ponderado por `length_s` (tiempo en segundos). Este cálculo nos dará el tiempo mínimo de viaje desde el mismo nodo de origen hasta cualquier otro punto de la red, permitiéndonos identificar las rutas más rápidas.

Es importante recordar que el camino más corto en distancia no siempre es el más rápido. Factores como los límites de velocidad, el tipo de carretera o la congestión pueden hacer que una ruta más larga en kilómetros sea más eficiente en tiempo.

In [25]:
dist_tiempo = cg.sssp(G_speed, nodo_origen)
max_float32_val = np.finfo(np.float32).max
dist_tiempo_validas = dist_tiempo['distance'].loc[dist_tiempo['distance'] < max_float32_val]
print('\nDistribución de tiempos (segundos) desde el nodo origen:')
print(dist_tiempo_validas.describe()[1:])
print(f'\nTiempo máximo alcanzable: {dist_tiempo_validas.max()/3600:.1f} horas')


Distribución de tiempos (segundos) desde el nodo origen:
mean     7424.899902
std      4666.250488
min         0.000000
25%      4484.137695
50%      6451.895508
75%      9064.045410
max     31424.103516
Name: distance, dtype: float64

Tiempo máximo alcanzable: 8.7 horas


El cálculo del SSSP con pesos de tiempo nos proporciona una distribución de los tiempos mínimos de viaje desde el nodo de origen. La media de 7424.9 segundos (aproximadamente 2.06 horas) y un tiempo máximo alcanzable de 8.7 horas, nos dan una perspectiva temporal de la red. Estos valores son cruciales para entender la accesibilidad y la eficiencia de la conectividad desde el nodo de origen, especialmente para aplicaciones donde el tiempo es el factor más crítico.

### Preparación de los resultados para visualización

El resultado del SSSP es una tabla `(vertex, distance, predecessor)`. Sin embargo, para visualizarlo geográficamente, debemos cruzarlo con las coordenadas de los nodos a través del `node_graph_map`.

In [26]:
# Unimos las distancias con las coordenadas geográficas
viz_df = road_nodes.merge(node_map, on='node_id')
viz_df = viz_df.merge(dist_tiempo, left_on='graph_id', right_on='vertex')

# Filtramos nodos no alcanzables e invertimos la escala
# Distancia menor -> valor más alto -> color más brillante
viz_df = viz_df[viz_df['distance'] < max_float32_val].copy()
viz_df['tiempo_inv'] = viz_df['distance'].pow(1/2).mul(-1)

print(f'Nodos con ruta válida: {len(viz_df):,}')
viz_df[['node_id', 'east', 'north', 'distance', 'tiempo_inv']].head()

Nodos con ruta válida: 3,048,671


,node_id,east,north,distance,tiempo_inv
0,id30401FD3-2046-40FE-AA54-0FE4F1AD20D6,380442.00000,857793.0000,24031.281250,-155.020264
1,idB5CC8A75-765D-4847-B9C8-16D77B803818,380434.00000,857803.0000,24032.250000,-155.023392
2,id0E8010C2-F329-4128-BC17-1EC645142A01,381577.59375,857983.8125,24045.074219,-155.064743
3,id4F8B9D66-5C63-49F6-8CE5-CF5802F0F387,380828.15625,858005.3125,24077.212891,-155.168335
4,id7298D90F-3290-44AD-B8BE-BCDF83190D3B,381088.81250,858156.8125,24083.847656,-155.189713


## 7. Visualización interactiva

Para esta última sección, haremos uso de la herramienta Graphistry para transformar los resultados numéricos del SSSP en un mapa interactivo donde el color de cada nodo indica el tiempo de viaje desde el punto de origen. Los nodos más brillantes están más cerca; los más oscuros están más lejos.

### Configuración de Graphistry

In [27]:
!pip install graphistry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 31.6 MB/s eta 0:00:00


In [51]:
import graphistry

# Regístrate en hub.graphistry.com y reemplaza con tus credenciales
# graphistry.register(api=3, username='USUARIO', password='PASSWORD')

### Visualización de nodos por tiempo de viaje

Graphistry funciona de manera óptima con `DataFrames` de pandas para cargar los datos de nodos y aristas. Aunque cuGraph y cuDF ofrecen un rendimiento superior en la GPU para el cálculo de algoritmos, para la fase de visualización es necesario convertir nuestros `DataFrames` de cuDF a pandas. Esta conversión permite que Graphistry acceda y procese los datos de manera compatible con su API.

Además, dado que estamos trabajando con una red de millones de nodos y aristas, es inviable enviar el grafo completo a la plataforma de Graphistry, especialmente en un entorno de taller. Para evitar errores de 'Request Entity Too Large' o problemas de conexión SSL, aplicaremos un muestreo significativo. Esto reduce el conjunto de datos a un tamaño manejable (por ejemplo, 50,000 nodos y 50,000 aristas), lo que permite que la visualización cargue efectivamente y sea interactiva sin sobrecargar los recursos.

In [29]:
viz_sample = viz_df.to_pandas()
edges_sample = road_graph.to_pandas()

print(f'Nodos en la visualización: {len(viz_sample):,}')
print(f'Aristas en la visualización: {len(edges_sample):,}')

Nodos en la visualización: 3,048,671
Aristas en la visualización: 7,218,169


In [47]:
N_VIZ = 50_000   # Limite para la visualización

viz_sampled_for_plot = viz_sample.sample(min(N_VIZ, len(viz_sample)), random_state=42)
edges_sampled_for_plot = edges_sample.sample(min(N_VIZ, len(edges_sample)), random_state=42)

(
    graphistry
    .nodes(viz_sampled_for_plot, 'node_id', x='east', y='north', color='tiempo_inv')
    .edges(edges_sampled_for_plot, 'src', 'dst', weight='length')
    .settings(url_params={
        'play': 0,
        'pointSize': 0.3
    })
    .plot()
)

En este mapa interactivo, cada punto representa un nodo (una intersección de carretera) y las líneas representan las aristas (tramos de carretera). El color de cada nodo representa la accesibilidad desde nuestro punto de origen (el nodo de mayor grado).

Los nodos se colorean según la variable `tiempo_inv`, que es una versión inversa y reescalada del tiempo mínimo de viaje desde el nodo de origen. Esto significa que los nodos brillantes (tonos amarillo/verde) están más cerca del origen en términos de tiempo de viaje, mientras que los nodos oscuros (tonos morado/negro) representan nodos que están más lejos del origen en tiempo de viaje.

### Construcción del Árbol de Caminos Mínimos

Una vez calculados los caminos mínimos desde el nodo de origen a todos los demás, el resultado del SSSP (`dist_tiempo`) no solo nos proporciona la distancia o el tiempo mínimo, sino también el `predecessor` (predecesor) de cada nodo en ese camino óptimo. Este campo es la clave para reconstruir el Árbol de Caminos Mínimos, una estructura arbórea que conecta el nodo de origen con cada nodo alcanzable a través de su ruta más eficiente. Visualizar este árbol nos permite entender la topología de las rutas óptimas y cómo se distribuye la accesibilidad desde el punto de partida seleccionado.

In [40]:
filtered_tree_data = dist_tiempo[
    # Filtrar los nodos alcanzables
    (dist_tiempo['distance'] < max_float32_val) &
    # Filtrar la fila del nodo origen ya que no es una arista del árbol
    (dist_tiempo['predecessor'] != -1)
]

tree_edges = filtered_tree_data[['vertex', 'predecessor']].to_pandas()
tree_edges.columns = ['dst', 'src'] # 'vertex' es el destino, 'predecessor' es el origen
print(f'Aristas en el árbol de caminos mínimos: {len(tree_edges):,}')

Aristas en el árbol de caminos mínimos: 3,048,670


In [41]:
tree_edges.head()

,dst,src
1,1252792,423201
2,1606375,1311665
3,1826781,1963186
4,1990110,1652530
5,2122903,2727979


El DataFrame `tree_edges` ahora contiene la información precisa de las 3,048,670 aristas que forman el árbol de caminos mínimos desde nuestro nodo de origen. Cada fila de este DataFrame representa una conexión directa en la ruta más rápida hacia un nodo específico. Hemos filtrado cuidadosamente los nodos inalcanzables y el propio nodo de origen (que no tiene un predecesor en el árbol), asegurando que solo las aristas válidas del árbol sean representadas. Este conjunto de aristas puede graficarse para observar cómo estas rutas óptimas se ramifican y cubren la red, revelando la estructura de accesibilidad más rápida.

In [50]:
N_TREE_VIZ = 10 # Limite para la visualización del árbol

tree_sample = tree_edges.sample(min(N_TREE_VIZ, len(tree_edges)), random_state=42)
viz_sampled_for_tree_plot = viz_sample.sample(min(N_TREE_VIZ, len(viz_sample)), random_state=42)

(
    graphistry
    .nodes(viz_sampled_for_tree_plot, 'node_id', x='east', y='north', color='tiempo_inv') # Atributos de nodo pasados directamente aquí
    .edges(tree_sample, 'src', 'dst')
    .settings(url_params={'play': 0, 'pointSize': 0.5, 'edgeOpacity': 0.3})
    .plot()
)

Esta visualización presenta un subgrafo especial de la red completa: el árbol de caminos mínimos desde el nodo de origen. A diferencia de la visualización anterior, donde las aristas representaban todas las conexiones viales muestreadas, aquí solo se muestran las aristas que forman parte de las rutas más rápidas desde el punto de partida seleccionado. Cada arista en este árbol apunta desde un nodo `src` a un `dst` que está un paso más cerca del origen en el camino mínimo.

## 8. Actividad

**Pregunta 1:** Al analizar los grados de los nodos, vimos que todos son múltiplos de 2. ¿Por qué es así en un grafo no dirigido? ¿Qué significaría que un nodo tuviera grado 1 en esta red de carreteras? ¿Y grado 0?

**Pregunta 2:** Ejecutamos el SSSP con dos grafos distintos: uno con pesos de longitud y otro con pesos de tiempo de viaje. Puede que el camino más corto en distancia no sea el mismo que el más rápido. ¿En qué situación real podría ocurrir que el camino más corto en kilómetros sea más lento que una ruta más larga? ¿En qué situaciones podría dársele prioridad a uno o al otro al momento de diseñar una ruta?

**Pregunta 3:** Retomemos la pregunta inicial: *¿desde dónde conviene partir si quiero llegar rápido a todos los demás lugares de la red?*

En el taller elegimos el nodo de mayor grado como punto de partida. Pero "conviene partir" puede interpretarse de varias maneras:

- Minimizar el tiempo máximo hasta cualquier otro punto (*minmax*).
- Minimizar la suma total de tiempos a todos los demás.
- Minimizar la distancia al punto más lejano de la red.

¿Estas tres interpretaciones darían el mismo nodo óptimo? ¿Hay una que te parezca más relevante para, por ejemplo, ubicar un centro de distribución logístico? Justifica tu razonamiento ejemplificando cómo las respuestas a estos enfoques podrían variar y en qué casos podría ser conveniente cada uno.